[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astronerd21/Deep-Learning-Oil-Slick-Detection/blob/main/Lookalike_Analysis.ipynb)

# False Positive / Lookalike Analysis
**Goal:** Inspect "NO OIL" (0) samples that the model incorrectly predicted as "OIL" (1). This notebook performs an automated evaluation to identify false positives and visualizes their radar signatures.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Install dependencies
!pip install rasterio

# Clone the repo to get the src/ folder structure
%cd /content
!rm -rf Deep-Learning-Oil-Slick-Detection
!git clone https://github.com/astronerd21/Deep-Learning-Oil-Slick-Detection.git
import sys
sys.path.append('/content/Deep-Learning-Oil-Slick-Detection')

In [ ]:
import os

# Setup directories
os.makedirs('/content/data', exist_ok=True)

# Extract satellite images from Google Drive
if not os.path.exists('/content/data/images_s1'):
    print("Extracting satellite images...")
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-00.tar -C /content/data/
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-01.tar -C /content/data/
else:
    print("Images already extracted.")

In [ ]:
# AUTOMATIC EVALUATION: This generates the required text file for us
%cd /content/Deep-Learning-Oil-Slick-Detection

# 1. Clean datasets
!PYTHONPATH=. python3 -m src.clean_dataset \
    --data-dir "/content/data/images_s1" \
    --splits-in-dir "/content/drive/MyDrive/OilSlickProject/data/OilSlick/splits/random"

# 2. Run Evaluation to generate false_positives_val_clean.txt
!PYTHONPATH=. python3 -m src.evaluate \
    --data-dir "/content/data/images_s1" \
    --model "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/best_cleaned_model.pt" \
    --split "/content/cleaned_splits/val_clean.txt" \
    --backbone resnet18

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

data_dir = Path("/content/data/images_s1/")
fp_list_path = Path("/content/evaluation_outputs/false_positives_val_clean.txt")

with open(fp_list_path, "r") as f:
    fp_filenames = [line.strip() for line in f if line.strip()]

top_10_fps = fp_filenames[:10]

def load_and_normalize_sar(filepath: Path) -> np.ndarray:
    with rasterio.open(filepath) as src:
        data = src.read().astype(np.float32)
    for c in range(data.shape[0]):
        channel_mean = np.mean(data[c])
        channel_std = np.std(data[c])
        if channel_std > 0:
            data[c] = (data[c] - channel_mean) / channel_std
    return data

num_samples = len(top_10_fps)
fig, axes = plt.subplots(num_samples, 4, figsize=(20, 4 * num_samples))
fig.suptitle("Lookalike Analysis: Normalized Radar Signatures & Histograms", fontsize=16, y=1.02)

for idx, filename in enumerate(top_10_fps):
    filepath = data_dir / filename
    data = load_and_normalize_sar(filepath)
    
    # Plotting code remains the same...
    axes[idx, 0].imshow(data[0], cmap='gray'); axes[idx, 0].set_title(f"VV | {filename}"); axes[idx, 0].axis('off')
    axes[idx, 1].hist(data[0].ravel(), bins=50, color='blue', alpha=0.7); axes[idx, 1].set_title("VV Hist")
    axes[idx, 2].imshow(data[1], cmap='gray'); axes[idx, 2].set_title(f"VH | {filename}"); axes[idx, 2].axis('off')
    axes[idx, 3].hist(data[1].ravel(), bins=50, color='orange', alpha=0.7); axes[idx, 3].set_title("VH Hist")

plt.tight_layout()
plt.show()